In [1]:
from collections import defaultdict
import numpy as np
import pandas as pd
import random
import re
from tqdm import tqdm
import torch
from transformers import LlamaTokenizer, LlamaForCausalLM
import os
import pickle
import matplotlib.pyplot as plt

os.environ["REQUESTS_CA_BUNDLE"] = "/etc/ssl/certs/ca-certificates.crt"
os.environ["SSL_CERT_FILE"] = "/etc/ssl/certs/ca-certificates.crt"

import random
import json

random.seed(42)
np.random.seed(42)

device = "cuda:0"
STORAGE_DIR = "cache/"

/mnt/hdd_drive/eduard/conda_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = LlamaForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf",
                                         cache_dir="/mnt/hdd_drive/huggingface/hub/",
                                         torch_dtype=torch.float16
                                       #  token=MY_TOKEN
                                        )
tokenizer = LlamaTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf",
                                            cache_dir="/mnt/hdd_drive/huggingface/hub/"
                                     #    token=MY_TOKEN
                                        )
model = model.to(device)

Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  7.62it/s]


In [3]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (no

In [4]:
from collections import OrderedDict 

class NewModel(torch.nn.Module):
    def __init__(self, model, *args):
        super().__init__(*args)
        self.selected_out = OrderedDict()

        self.pretrained = model
        self.fhooks = []

        for i in range(32):
            self.fhooks.append(self.pretrained.model.layers[i].self_attn.q_proj
                .register_forward_hook(self.forward_hook("query_vec_" + str(i))))
            self.fhooks.append(self.pretrained.model.layers[i].self_attn.k_proj
                .register_forward_hook(self.forward_hook("key_vec_" + str(i))))
        
    
    def forward_hook(self, layer_name):
        def hook(module, input, output):
            self.selected_out[layer_name] = output.cpu()
        return hook

    def forward(self, x):        
        out = self.pretrained(**x)
        return out, self.selected_out
    
newmodel = NewModel(model)

In [5]:
def permute_options(options, label, seed):
    """
    A pseudo-random shuffle of the answer <<options>> that also returns new correct answer
        parameters:
                options --- list of options
                label   --- correct answer
                seed    --- random seed
                
    Options 'E' and 'F' are special and must remain at their places
    """
    aaa = np.arange(len(options) - 2)
    np.random.seed(seed)
    np.random.shuffle(aaa)
    
    new_label = chr(ord('A') + aaa[ord(label) - ord('A')])
    new_options = {}
    for option in ['A', 'B', 'C', 'D']: 
        new_options[chr(aaa[ord(option) - ord('A')] + ord('A'))] = options[option]
   
    new_options['E'] = options['E']  
    new_options['F'] = options['F']

    return dict(sorted(new_options.items())), new_label

def format_prompt(elem):
    text = ""
    if "context" in elem.keys():
        text += "Context: " + elem["context"] + "\n"
    text += "Question: " + elem["question"] + "\nOptions:\n"
    for option in elem['choices'].keys():
        text += option + ". " + str(elem['choices'][option]) + "\n"   
    text += "Answer: " + elem["answer"] + "\n"
    return text
    
def angular_dist(vec_a, vec_b):
    # Without normalization it works much faster and yields better results for some unknown reason
    return torch.sum(vec_a * vec_b) #/ (vec_a.norm(p=2) * vec_b.norm(p=2))

In [6]:
def print_results(true_labels, kv_predicts, baseline):
    mn = np.zeros((32,32))
    for i in range(32):
        for j in range(32):
            mn[i][j] = np.mean(kv_predicts[i][j] == true_labels)
    print("Q-K score", np.max(mn), np.argmax(mn) // 32, np.argmax(mn) % 32)
    print("Baseline score", np.mean(np.array(true_labels) == baseline))
    return (np.mean(np.array(true_labels) == baseline), np.max(mn)), (np.argmax(mn) // 32, np.argmax(mn) % 32)

def print_results_test(true_labels, kv_predicts, baseline):
    mn = np.mean(kv_predicts == true_labels)
    print("Q-Kresults", mn)
    print("Baseline score", np.mean(np.array(true_labels) == baseline))
    return (np.mean(np.array(true_labels) == baseline), mn)

def save_results(dataset, shots):
    np.save('cache/{}_10k_llama2_{}-shot_predictions_q_k.npy'.format(dataset, shots), predicted_labels)
 #   np.save('cache/{}_10k_llama2_{}-shot_raw_scores_q_k.npy'.format(dataset, shots), predicted_raw)
   # np.save('cache/{}_10k_llama2_baseline_{}-shot_logits.npy'.format(dataset, shots), logits_options)
   # np.save('cache/{}_10k_llama2_baseline_{}-shot_predictions.npy'.format(dataset, shots), next_token_labels)
  #  np.save('cache/{}_10k_true_labels.npy'.format(dataset), true_labels)

In [7]:
def do_calc_dev(data, prompt='', samples_range=range(10000), permute=False, return_logits=True, num=0):
    """
    Parameters:
        data      ------- self-explanatory
        prompt      ----- Here go examples in case of the Few-Shot prompting. For Zero-shot leave it empty.
        samples_range --- Container with numbers of samples to be considered 
        permute     ----- Specifies if a permutation of answer options is required
        return_logits  -- Set True if you need not only the predicted labels, but also row similarities and baseline logits
                                    for every possible option A-F
    
    """
    true_labels = []
    predicted_labels, predicted_raw = [], []
    next_token_labels, logits_options = [], []

    for LAYER in range(32):
        predicted_labels.append([])
        predicted_raw.append([])

        for HEAD in range(32):
            predicted_labels[LAYER].append([])
            predicted_raw[LAYER].append([])

    for EXMPL in tqdm(samples_range):
        prompt2 = ""
        for i in range(len(num)):
            prompt2 += format_prompt(data[num[i]])
        """
        Assembling the prompt from different parts: Examples (if any) + Context + Question + Options + Finisher
        """    
        if 'context' in data[EXMPL].keys():    # Some quesions are given without context
            encodinds_context_q = tokenizer(prompt2 + "Context: " + data[EXMPL]['context'] + "\nQuestion: " + \
                                            data[EXMPL]['question'] + "\nOptions:\n", return_tensors="pt")
        else:
            encodinds_context_q = tokenizer(prompt2 + "Question: " + data[EXMPL]['question'] + "\nOptions:\n", 
                                            return_tensors="pt")
            
        num_q = encodinds_context_q["input_ids"].shape[-1] - 1

        encodings_answ, options_answ = [], []
        
        """ 
        For some experiments we need to permute answer options
        """
        options_raw, answer_raw = data[EXMPL]['choices'], data[EXMPL]['answer']
        if permute:
            options_raw, answer_raw = permute_options(options_raw, answer_raw, EXMPL)
        
        for option in options_raw.keys():
            options_raw[option] = str(options_raw[option])            
            encodings_answ.append(tokenizer(option + ". " + options_raw[option] + "\n", return_tensors="pt"))
            if len(options_answ) == 0:
                options_answ.append(int(num_q + encodings_answ[-1]["input_ids"].shape[-1] - 1))
            else:
                options_answ.append(int(options_answ[-1] + encodings_answ[-1]["input_ids"].shape[-1] - 1))

        encodings_answ.append(tokenizer("Answer:", return_tensors="pt"))
        inputs = {
            "input_ids" : torch.cat([encodinds_context_q["input_ids"]] + [x["input_ids"][..., 1:] for x in encodings_answ], 1).to(device),
            "attention_mask" : torch.cat([encodinds_context_q["attention_mask"]] + [x["attention_mask"][..., 1:] for x in encodings_answ], 1).to(device)
        }
        
        """
        If promt format was changed use this to debug:
        
        print(inputs)
        for i in range(len(options_answ)):
            print(inputs["input_ids"][..., options_answ[i]]) # <<<<<< This must be aligned
        print("\n\n", inputs["input_ids"][..., -1])
        """
        with torch.no_grad():
            outputs = newmodel(inputs)

        true_labels.append(answer_raw)

        for LAYER in range(32):
            for HEAD in range(32):
                predicts = np.zeros(len(options_answ))
                for i in range(len(options_answ)):
                    predicts[i] = angular_dist(outputs[1]["query_vec_" + str(LAYER)][0][-1][HEAD * 128:(HEAD + 1) * 128], 
                                                                                    outputs[1]["key_vec_" + str(LAYER)][0][options_answ[i]][HEAD * 128:(HEAD + 1) * 128])

                predicted_labels[LAYER][HEAD].append(chr(np.argmax(predicts) + ord('A')))
                if return_logits:
                    predicted_raw[LAYER][HEAD].append(predicts)

        """
        This part is for the baseline prediction --- that is an A-F letter which is the most probable next token after prompt.
        """
        logits = outputs[0].logits.detach().cpu()
        logits = logits[:, -1, :]
        logits_full = logits.squeeze(0)
        logits_reduced = logits_full[np.array([319, 350, 315, 360, 382, 383])].numpy() ## LLAMA2 tokens for letters 'A'-'F'
        
        if return_logits:
            logits_options.append(logits_reduced)
        
        next_token_labels.append(chr(ord('A') + np.argmax(logits_reduced)))
        """
        end of baseline part
        """

    for LAYER in range(32):
        predicted_labels[LAYER] = np.array(predicted_labels[LAYER])
        if return_logits:
            predicted_raw[LAYER] = np.array(predicted_raw[LAYER])

    predicted_labels = np.stack(predicted_labels)
    
    if return_logits:
        predicted_raw = np.stack(predicted_raw)
        logits_options = np.stack(logits_options)

        return true_labels, (predicted_labels, predicted_raw), (next_token_labels, logits_options)
    else:
        return true_labels, predicted_labels, next_token_labels

In [8]:
def do_calc_eval(data, head=(0,0), prompt='', samples_range=range(10000), permute=False, num=[0,]):
    """
    Parameters:
        data      ------- self-explanatory
        head          --- Pair (#LAYER, #HEAD),
        prompt      ----- Here go examples in case of the Few-Shot prompting. For Zero-shot leave it empty.
        samples_range --- Container with numbers of samples to be considered 
        permute     ----- Specifies if a permutation of answer options is required    
    """
    true_labels = []
    predicted_labels = []
    next_token_labels = []

    for EXMPL in tqdm(samples_range):
        prompt2 = ""
        for i in range(len(num)):
            prompt2 += format_prompt(data[num[i]])
        """
        Assembling the prompt from different parts: Examples (if any) + Context + Question + Options + Finisher
        """    
        if 'context' in data[EXMPL].keys():    # Some quesions are given without context
            encodinds_context_q = tokenizer(prompt2 + "Context: " + data[EXMPL]['context'] + "\nQuestion: " + \
                                            data[EXMPL]['question'] + "\nOptions:\n", return_tensors="pt")
        else:
            encodinds_context_q = tokenizer(prompt2 + "Question: " + data[EXMPL]['question'] + "\nOptions:\n", 
                                            return_tensors="pt")
            
        num_q = encodinds_context_q["input_ids"].shape[-1] - 1
        encodings_answ, options_answ = [], []
        
        """ 
        For some experiments we need to permute answer options
        """
        options_raw, answer_raw = data[EXMPL]['choices'], data[EXMPL]['answer']
        if permute:
            options_raw, answer_raw = permute_options(options_raw, answer_raw, EXMPL)
        
        for option in options_raw.keys():
            options_raw[option] = str(options_raw[option])            
            encodings_answ.append(tokenizer(option + ". " + options_raw[option] + "\n", return_tensors="pt"))
            if len(options_answ) == 0:
                options_answ.append(int(num_q + encodings_answ[-1]["input_ids"].shape[-1] - 1))
            else:
                options_answ.append(int(options_answ[-1] + encodings_answ[-1]["input_ids"].shape[-1] - 1))

        encodings_answ.append(tokenizer("Answer:", return_tensors="pt"))
        inputs = {
            "input_ids" : torch.cat([encodinds_context_q["input_ids"]] + [x["input_ids"][..., 1:] for x in encodings_answ], 1).to(device),
            "attention_mask" : torch.cat([encodinds_context_q["attention_mask"]] + [x["attention_mask"][..., 1:] for x in encodings_answ], 1).to(device)
        }
        
        """
        If promt format was changed use this to debug:
        
        print(inputs)
        for i in range(len(options_answ)):
            print(inputs["input_ids"][..., options_answ[i]]) # <<<<<< This must be aligned
        print("\n\n", inputs["input_ids"][..., -1])
        """
        with torch.no_grad():
            outputs = newmodel(inputs)

        true_labels.append(answer_raw)
        
        LAYER, HEAD = head[0], head[1]
        
        predicts = np.zeros(len(options_answ))
        for i in range(len(options_answ)):
            predicts[i] = angular_dist(outputs[1]["query_vec_" + str(LAYER)][0][-1][HEAD * 128:(HEAD + 1) * 128], 
                                                                            outputs[1]["key_vec_" + str(LAYER)][0][options_answ[i]][HEAD * 128:(HEAD + 1) * 128])

        predicted_labels.append(chr(np.argmax(predicts) + ord('A')))

        """
        This part is for the baseline prediction --- that is an A-F letter which is the most probable next token after prompt.
        """
        logits = outputs[0].logits.detach().cpu()
        logits = logits[:, -1, :]
        logits_full = logits.squeeze(0)
        logits_reduced = logits_full[np.array([319, 350, 315, 360, 382, 383])].numpy() ## LLAMA2 tokens for letters 'A'-'F'
        
        next_token_labels.append(chr(ord('A') + np.argmax(logits_reduced)))
        """
        end of baseline part
        """

    return np.array(true_labels), np.array(predicted_labels), np.array(next_token_labels)

In [9]:
def compute_OUR_metric(target, predictions, target_permuted, predictions_permuted):
    our_metric_value = np.mean((np.array(target) == np.array(predictions)) * (np.array(target_permuted) == np.array(predictions_permuted)))
    return our_metric_value

### Main calculations

In [ ]:
DATASETS = ['halu_dialogue_new', 'hellaswag_10k_new', 'cosmosqa_10k_new']
RERUNS = 5
for dataset in DATASETS:
    json_file = 'llm_uncertaint_eval/ + ' dataset + '_new.json'

    with open(json_file) as json_data:
        data = json.load(json_data)
    metrics_to_report = [{'baseline_accs' : [], 
                         'baseline_ours' : [],
                         'qk_score_ours' : [],
                         'qk_score_accs' : []} for x in range(6)]
    
    idx_cache = []
    
    np.random.seed(42)
    for i in range(RERUNS):
        for EXAMPLES in range(1, 6):
            idx = np.random.choice(500, size=(EXAMPLES), replace=False).tolist()
            while idx in idx_cache:
                idx = np.random.choice(500, size=(EXAMPLES), replace=False).tolist()
            idx_cache.append(idx)
            
            true_labels, predicted_labels, next_token_labels = do_calc_dev(data, '', range(500), False, False, idx)
            print(idx)
            _, head = print_results(true_labels, predicted_labels, next_token_labels)
        
            true_labels2, predicted_labels2, next_token_labels2 = do_calc_dev(data, '', range(500), True, False, idx)
            _, head2 = print_results(true_labels2, predicted_labels2, next_token_labels2)
        
            true_labels, predicted_labels, next_token_labels = do_calc_eval(data, head, prompt='', samples_range=range(500, 10000), permute=False, num=idx)
            metrics = print_results_test(true_labels, predicted_labels, next_token_labels)
        
            true_labels2, predicted_labels2, next_token_labels2 = do_calc_eval(data, head2, prompt='', samples_range=range(500, 10000), permute=True, num=idx)
            metrics2 = print_results_test(true_labels2, predicted_labels2, next_token_labels2)
        
            metrics_to_report[EXAMPLES]['baseline_accs'].append(metrics[0])
            metrics_to_report[EXAMPLES]['baseline_ours'].append(compute_OUR_metric(true_labels, next_token_labels, true_labels2, next_token_labels2))
            metrics_to_report[EXAMPLES]['qk_score_accs'].append(metrics[1])
            metrics_to_report[EXAMPLES]['qk_score_ours'].append(compute_OUR_metric(true_labels, predicted_labels, true_labels2, predicted_labels2))
    
            with open(dataset + "_metrics.pkl", "wb") as file:
                pickle.dump(metrics_to_report, file)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [02:11<00:00,  3.79it/s]


[361]
Q-K score 0.408 15 4
Baseline score 0.412


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [02:09<00:00,  3.85it/s]


Q-K score 0.416 15 4
Baseline score 0.378


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9500/9500 [18:22<00:00,  8.61it/s]


...

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [02:06<00:00,  3.96it/s]


[207]
Q-K score 0.406 14 24
Baseline score 0.402


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [02:06<00:00,  3.96it/s]


Q-K score 0.42 14 4
Baseline score 0.368


 57%|█████████████████████████████████████████████████████████████████████████████████████████████████▌                                                                        | 5451/9500 [09:52<09:30,  7.10it/s]

In [31]:
metrics_to_report

[{'baseline_accs': [],
  'baseline_ours': [],
  'qk_score_ours': [],
  'qk_score_accs': []},
 {'baseline_accs': [0.37473684210526315],
  'baseline_ours': [0.19357894736842104],
  'qk_score_ours': [0.216],
  'qk_score_accs': [0.3894736842105263]},
 {'baseline_accs': [],
  'baseline_ours': [],
  'qk_score_ours': [],
  'qk_score_accs': []},
 {'baseline_accs': [],
  'baseline_ours': [],
  'qk_score_ours': [],
  'qk_score_accs': []},
 {'baseline_accs': [],
  'baseline_ours': [],
  'qk_score_ours': [],
  'qk_score_accs': []},
 {'baseline_accs': [],
  'baseline_ours': [],
  'qk_score_ours': [],
  'qk_score_accs': []}]